# 🐍 ViperScope — Exoplanet Transit Detection Notebook
**Bio-inspired transit detection · False-positive triage · Survey-scale search**

| Section | What it does |
|---------|-------------|
| **1. Config & Setup** | Single-source constants and dependency install |
| **2. Algorithm** | Full ViperScope / PitViper definition |
| **3. Validation** | Test on known planet (WASP-17b) |
| **4. Survey** | Build target list + run on uninvestigated stars |
| **5. Ledger** | View all candidates found |
| **6. Investigate** | Deep-dive any candidate |


---
# Section 1 — Setup

In [ ]:
# NASA Rule 8 — all constants defined once here; never magic-numbered elsewhere.

# ── Dependency install ──────────────────────────────────────────────────────
import subprocess, sys

REQUIRED_PACKAGES = ('lightkurve', 'astropy', 'astroquery')  # Rule 8: single definition
MAX_INSTALL_ATTEMPTS = 1                                      # Rule 2: bounded loop

for pkg in REQUIRED_PACKAGES:                                 # Rule 2: fixed upper bound
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    assert result.returncode == 0, f'Failed to install {pkg}: {result.stderr}'  # Rule 5

print('All dependencies installed.')


In [ ]:
import warnings
import os
import json
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import lombscargle
from scipy.ndimage import median_filter

# NASA Rule 10 — treat warnings as errors in production.
# Switch to 'ignore' only when running interactively and you understand each warning.
warnings.filterwarnings('error', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='.*Matplotlib.*')   # plotting noise only

# ── NASA Rule 8: Single-source configuration ────────────────────────────────
# All numeric limits, thresholds, and paths are defined ONCE here.
# Never repeat these values elsewhere in the notebook.

# File-system layout
SURVEY_DIR   = 'viperscope_survey'
REPORTS_DIR  = os.path.join(SURVEY_DIR, 'reports')
TARGETS_FILE = os.path.join(SURVEY_DIR, 'target_list.json')
PROGRESS_FILE = os.path.join(SURVEY_DIR, 'progress.json')
LEDGER_FILE  = os.path.join(SURVEY_DIR, 'discovery_ledger.json')

# Survey search parameters
PERIOD_MIN_DAYS   = 0.5      # minimum orbital period to search
PERIOD_MAX_DAYS   = 15.0     # maximum orbital period to search
N_PERIOD_STEPS    = 800      # number of trial periods in phase-fold grid
SNR_THRESHOLD_PF  = 1.3      # phase-fold detection threshold
SNR_THRESHOLD_FP  = 7.0      # SNR required to pass false-positive test
MAX_DEPTH_FRAC    = 0.03     # 3 % — upper limit for planetary transit depth
MAX_SHAPE_RATIO   = 2.0      # ingress/egress symmetry limit
MAX_SECONDARY_DEP = 0.001    # secondary eclipse depth limit
MAX_EVEN_ODD_DIFF = 0.3      # even/odd transit asymmetry limit
MAX_DUR_FACTOR    = 1.5      # maximum duration relative to physical max

# Preprocessing
SAVGOL_WINDOW_DAYS  = 0.5    # Savitzky-Golay trend window in days
SAVGOL_POLYORDER    = 2      # polynomial order for detrending
SIGMA_CLIP          = 5      # sigma threshold for outlier removal
SAVGOL_MIN_WIN      = 5      # minimum Savitzky-Golay window (samples)

# Survey logistics
BATCH_SIZE          = 20     # stars processed per notebook run
MAX_TIC_PER_FIELD   = 80     # row limit per TIC cone search
MAG_MIN             = 6.0    # minimum TESS magnitude (avoid saturation)
MAG_MAX             = 13.0   # maximum TESS magnitude (signal-to-noise)
TEFF_MIN            = 4000   # minimum effective temperature (K)
TEFF_MAX            = 7000   # maximum effective temperature (K)
LOGG_MIN            = 3.5    # minimum log g (exclude giants)
TIC_QUERY_RADIUS_DEG = 1.0   # cone search radius in degrees
TIC_TIMEOUT_S        = 120   # astroquery timeout in seconds
TOI_TIMEOUT_S        = 20    # ExoFOP download timeout
N_TARGETS_DEFAULT    = 500   # default survey size

# Physical constants (SI)
G_SI   = 6.674e-11   # gravitational constant (m³ kg⁻¹ s⁻²)
M_SUN  = 1.989e30    # solar mass (kg)
R_SUN  = 6.957e8     # solar radius (m)
S_PER_DAY = 86400    # seconds per day

# Known planets used for cross-matching (name → period in days)
KNOWN_PLANETS: dict[str, float] = {
    'WASP-17' : 3.7354,
    'WASP-39' : 4.0553,
    'HD 209458': 3.5247,
    'KELT-9'  : 1.4822,
    'WASP-76' : 1.8099,
    'WASP-121': 1.2749,
}

for d in (SURVEY_DIR, REPORTS_DIR):
    os.makedirs(d, exist_ok=True)

print('Imports and configuration ready.')


---
# Section 2 — ViperScope Algorithm
Run this cell once to load all functions. Re-run if you edit anything.

In [ ]:
# ── Data ingestion ──────────────────────────────────────────────────────────
# NASA Rule 4: every function ≤ 60 lines.
# NASA Rule 5: two assertions per function.
# NASA Rule 9: inputs are never mutated in-place.

def load_from_fits(fits_path: str):
    """
    Load a TESS light curve from a local FITS file.

    Parameters
    ----------
    fits_path : str  Path to a TESS FITS light-curve file.

    Returns
    -------
    tuple (time, flux, flux_err, sector, cadence) or None on failure.
    """
    # Rule 5 — entry assertion
    assert isinstance(fits_path, str) and len(fits_path) > 0, 'fits_path must be a non-empty string'

    from astropy.io import fits as af
    from astropy.table import Table

    with af.open(fits_path) as hdul:
        data = Table(hdul[1].data)

    time = np.array(data['TIME'], dtype=float)

    flux_col = 'PDCSAP_FLUX' if 'PDCSAP_FLUX' in data.colnames else 'FLUX'
    flux = np.array(data[flux_col], dtype=float)

    err_col = (
        'PDCSAP_FLUX_ERR' if 'PDCSAP_FLUX_ERR' in data.colnames
        else 'FLUX_ERR'   if 'FLUX_ERR'        in data.colnames
        else None
    )
    flux_err = np.array(data[err_col], dtype=float) if err_col else np.full_like(flux, np.nan)

    valid_mask = np.isfinite(time) & np.isfinite(flux)
    time     = time[valid_mask].copy()       # Rule 9: new array, no alias
    flux     = flux[valid_mask].copy()
    flux_err = flux_err[valid_mask].copy()

    median_flux = np.nanmedian(flux)
    assert median_flux != 0.0, 'Median flux is zero — cannot normalise'

    flux     = flux     / median_flux
    flux_err = flux_err / median_flux

    cadence = float(np.median(np.diff(time)))
    # Rule 5 — exit assertion
    assert len(time) == len(flux) == len(flux_err), 'Output arrays have inconsistent lengths'
    return time, flux, flux_err, 0, cadence


def download_star(target: str):
    """
    Download a TESS light curve via Lightkurve.

    Parameters
    ----------
    target : str  Target identifier (TIC ID, common name, etc.).

    Returns
    -------
    tuple (time, flux, flux_err, sector, cadence) or None if unavailable.
    """
    assert isinstance(target, str) and len(target) > 0, 'target must be a non-empty string'  # Rule 5

    try:
        import lightkurve as lk

        search_result = lk.search_lightcurve(target, mission='TESS', author='SPOC')
        if len(search_result) == 0:
            search_result = lk.search_lightcurve(target, mission='TESS')
        if len(search_result) == 0:
            return None  # Rule 7: explicit None return

        lc = search_result[0].download().remove_nans().normalize()
        time     = lc.time.value.astype(float).copy()       # Rule 9
        flux     = lc.flux.value.astype(float).copy()
        flux_err = (
            lc.flux_err.value.astype(float).copy()
            if hasattr(lc, 'flux_err')
            else np.full_like(flux, np.std(flux) * 0.1)
        )
        sector  = int(getattr(lc, 'sector', 0))
        cadence = float(np.median(np.diff(time)))

        # Robust 5-sigma clip (PitViper thermal sensing)
        median_f = np.median(flux)
        sigma_f  = np.std(flux)
        keep     = np.abs(flux - median_f) < SIGMA_CLIP * sigma_f

        time     = time[keep].copy()      # Rule 9
        flux     = flux[keep].copy()
        flux_err = flux_err[keep].copy()

        assert len(time) > 10, f'Too few cadences after sigma-clip: {len(time)}'  # Rule 5
        return time, flux, flux_err, sector, cadence

    except Exception as exc:
        print(f'    [WARN] {target}: {exc}')
        return None  # Rule 7


# ── Preprocessing (PitViper thermal background detector) ────────────────────

def preprocess_flux(time: np.ndarray, flux: np.ndarray,
                    window: float = SAVGOL_WINDOW_DAYS,
                    polyorder: int = SAVGOL_POLYORDER) -> np.ndarray:
    """
    Remove long-term trends from a flux time series using a Savitzky-Golay filter.

    Returns
    -------
    detrended : np.ndarray  Fractional residuals (flux / trend − 1).
    """
    assert len(time) == len(flux), 'time and flux must have the same length'  # Rule 5
    assert len(time) >= SAVGOL_MIN_WIN, f'Need at least {SAVGOL_MIN_WIN} cadences'

    from scipy.signal import savgol_filter

    time_arr = np.asarray(time, dtype=float)   # Rule 9: work on copies
    flux_arr = np.asarray(flux, dtype=float)

    finite_mask = np.isfinite(time_arr) & np.isfinite(flux_arr)
    time_clean = time_arr[finite_mask]
    flux_clean = flux_arr[finite_mask]

    dt = float(np.median(np.diff(time_clean)))
    win = max(SAVGOL_MIN_WIN, int(window / dt))
    if win % 2 == 0:
        win += 1
    win = min(win, len(flux_clean) - (1 if len(flux_clean) % 2 == 0 else 0))

    trend    = savgol_filter(flux_clean, win, polyorder)
    detrended = flux_clean / trend - 1.0

    assert len(detrended) == len(flux_clean), 'Detrended output length mismatch'  # Rule 5
    return detrended


# ── Period recovery ─────────────────────────────────────────────────────────

def phase_fold_search(energy: np.ndarray,
                      period_min: float,
                      period_max: float,
                      n_periods: int = N_PERIOD_STEPS,
                      snr_threshold: float = SNR_THRESHOLD_PF) -> dict:
    """
    Vectorised phase-fold period search with harmonic de-aliasing.

    Returns a dict with keys: period, snr, detection, periods, scores.
    """
    # Rule 5 — entry assertions
    assert period_min > 0 and period_max > period_min, 'Invalid period range'
    assert n_periods > 0, 'n_periods must be positive'

    periods = np.linspace(period_min, period_max, n_periods)  # Rule 3: fixed-size array
    scores  = np.zeros(n_periods)                              # Rule 3: pre-allocated

    # Rule 2: loop bounded by n_periods
    for i in range(n_periods):
        P     = periods[i]
        P_int = max(2, int(round(P * n_periods / period_max)))
        folded = np.zeros(P_int)   # Rule 3
        counts = np.zeros(P_int)
        idx    = np.arange(len(energy)) % P_int
        np.add.at(folded, idx, energy)
        np.add.at(counts, idx, 1)
        counts  = np.maximum(counts, 1)    # avoid division by zero
        folded /= counts
        mad     = np.median(np.abs(folded - np.median(folded)))
        scores[i] = (np.percentile(folded, 75) - np.min(folded)) / (mad + 1e-9)

    best_idx = int(np.argmax(scores))

    # Harmonic de-aliasing (check sub-harmonics up to 4×)
    # Rule 2: loop bounded by constant 4
    for divisor in (2, 3, 4):
        p_sub = periods[best_idx] / divisor
        if p_sub < periods[0]:
            continue
        nearest = int(np.argmin(np.abs(periods - p_sub)))
        within_5pct = abs(periods[nearest] - p_sub) / p_sub < 0.05
        strong_enough = scores[nearest] >= scores[best_idx] * 0.90
        if within_5pct and strong_enough:
            best_idx = nearest
            break

    snr = float(scores[best_idx] / (np.median(scores) + 1e-9))

    # Rule 5 — exit assertion
    assert np.isfinite(snr), 'SNR is not finite'
    return {'period': float(periods[best_idx]), 'snr': snr,
            'detection': snr >= snr_threshold,
            'periods': periods, 'scores': scores}


def lombscargle_confirm(energy: np.ndarray,
                        time_arr: np.ndarray,
                        period_min: float,
                        period_max: float) -> dict:
    """
    Confirm a candidate period using the Lomb-Scargle periodogram.

    Returns a dict with keys: period, snr, freqs, power.
    """
    assert len(energy) == len(time_arr), 'energy and time_arr must match'  # Rule 5
    assert period_min > 0 and period_max > period_min, 'Invalid period range'

    window     = max(3, len(energy) // 20)
    trend      = median_filter(energy, size=window, mode='nearest')
    detrended  = energy - trend
    detrended -= detrended.mean()

    freqs = np.linspace(1.0 / (period_max * 0.98),
                        1.0 / (period_min * 1.02), 5000)  # Rule 3: fixed size
    power = lombscargle(time_arr, detrended, freqs * 2 * np.pi, normalize=True)

    best_period = float(1.0 / freqs[np.argmax(power)])
    snr         = float(power.max() / (np.median(power) + 1e-9))

    assert np.isfinite(best_period) and best_period > 0, 'LS period is non-positive or NaN'  # Rule 5
    return {'period': best_period, 'snr': snr, 'freqs': freqs, 'power': power}


def fold_lightcurve(time: np.ndarray,
                    flux: np.ndarray,
                    period: float,
                    t0: float = None):
    """
    Phase-fold a light curve at a given period.

    Returns (phase, flux_sorted) — both sorted by phase.
    """
    assert len(time) == len(flux), 'time and flux must match'  # Rule 5
    assert period > 0, 'period must be positive'

    t_ref = float(time[0]) if t0 is None else float(t0)
    phase = ((time - t_ref) % period) / period   # Rule 9: new array
    phase = phase.copy()
    phase[phase > 0.5] -= 1.0

    sort_idx = np.argsort(phase)
    assert len(sort_idx) == len(time), 'Sort index length mismatch'  # Rule 5
    return phase[sort_idx], flux[sort_idx].copy()


def bin_folded(phase: np.ndarray, flux: np.ndarray, n_bins: int = 100):
    """
    Bin a phase-folded light curve into n_bins phase bins.

    Returns (bin_centres, bin_medians).
    """
    assert len(phase) == len(flux), 'phase and flux must match'  # Rule 5
    assert n_bins > 0, 'n_bins must be positive'

    edges     = np.linspace(-0.5, 0.5, n_bins + 1)  # Rule 3
    centres   = (edges[:-1] + edges[1:]) / 2.0
    medians   = np.full(n_bins, np.nan)               # Rule 3: pre-allocated

    # Rule 2: loop bounded by n_bins
    for b in range(n_bins):
        mask = (phase >= edges[b]) & (phase < edges[b + 1])
        if mask.any():
            medians[b] = float(np.median(flux[mask]))

    assert len(centres) == len(medians) == n_bins, 'Output length mismatch'  # Rule 5
    return centres, medians


# ── False positive triage ────────────────────────────────────────────────────

def measure_transit(phase: np.ndarray, flux: np.ndarray,
                    width: float = 0.1):
    """
    Measure the depth and shape of a transit near phase = 0.

    Returns a dict with depth, baseline, ingress_slope, egress_slope,
    or None if there are insufficient in-transit points.
    """
    assert len(phase) == len(flux), 'phase and flux must match'  # Rule 5

    in_transit  = np.abs(phase) < width / 2.0
    out_transit = np.abs(phase) > width

    if in_transit.sum() < 3 or out_transit.sum() < 3:
        return None  # Rule 7: explicit None

    baseline = float(np.median(flux[out_transit]))
    depth    = float((baseline - np.median(flux[in_transit])) / (baseline + 1e-9))

    first_half  = (phase > -width / 2.0) & (phase < 0.0)
    second_half = (phase > 0.0)          & (phase < width / 2.0)

    ingress = (
        abs(float(np.polyfit(phase[first_half],  flux[first_half],  1)[0]))
        if first_half.sum() > 2 else 0.0
    )
    egress = (
        abs(float(np.polyfit(phase[second_half], flux[second_half], 1)[0]))
        if second_half.sum() > 2 else 0.0
    )

    assert np.isfinite(depth), 'Transit depth is not finite'  # Rule 5
    return {'depth': depth, 'baseline': baseline,
            'ingress_slope': ingress, 'egress_slope': egress}


def check_secondary(time: np.ndarray, flux: np.ndarray, period: float) -> float:
    """Return the secondary eclipse depth at phase ±0.5 (0 = no secondary)."""
    assert len(time) == len(flux) and period > 0  # Rule 5

    phase, folded = fold_lightcurve(time, flux, period)
    near_secondary = np.abs(np.abs(phase) - 0.5) < 0.05
    out_of_transit = np.abs(phase) > 0.2

    if near_secondary.sum() < 2 or len(folded[out_of_transit]) < 5:
        return 0.0
    return float(np.median(folded[out_of_transit]) - np.median(folded[near_secondary]))


def check_even_odd(time: np.ndarray, flux: np.ndarray, period: float) -> float:
    """Return fractional even/odd depth difference (0 = symmetric)."""
    assert len(time) == len(flux) and period > 0  # Rule 5

    t0          = float(time[0])
    transit_num = np.floor((time - t0) / period).astype(int)
    phase       = ((time - t0) % period) / period
    phase       = phase.copy()
    phase[phase > 0.5] -= 1.0

    in_transit = np.abs(phase) < 0.05
    if in_transit.sum() < 6:
        return 0.0

    even_mask = in_transit & (transit_num % 2 == 0)
    odd_mask  = in_transit & (transit_num % 2 == 1)
    if even_mask.sum() < 2 or odd_mask.sum() < 2:
        return 0.0

    baseline   = float(np.median(flux[np.abs(phase) > 0.2]))
    even_depth = baseline - float(np.median(flux[even_mask]))
    odd_depth  = baseline - float(np.median(flux[odd_mask]))
    denom      = (abs(even_depth) + abs(odd_depth)) / 2.0 + 1e-9

    assert np.isfinite(denom) and denom > 0  # Rule 5
    return float(abs(even_depth - odd_depth) / denom)


def phys_max_duration(period_days: float) -> float:
    """
    Compute the maximum physical transit duration for a circular orbit
    around a Sun-like star (grazing geometry).

    Returns duration in days.
    """
    # NASA Rule 8: uses named constants defined in config block
    import math
    assert period_days > 0, 'period must be positive'  # Rule 5

    period_s  = period_days * S_PER_DAY
    semi_axis = (G_SI * M_SUN * period_s ** 2 / (4.0 * math.pi ** 2)) ** (1.0 / 3.0)
    orbital_v = 2.0 * math.pi * semi_axis / period_s   # m/s
    dur_s     = 2.0 * R_SUN / orbital_v

    assert dur_s > 0, 'Computed duration is non-positive'  # Rule 5
    return dur_s / S_PER_DAY


def run_fp_tests(time: np.ndarray, flux: np.ndarray,
                 period: float, snr: float,
                 cadence: float = 0.0208) -> dict:
    """
    Run all false-positive (FP) tests and return a disposition.

    Returns a dict with keys:
      disposition, tests, depth, transit_dur, n_fail, failed_tests, reason.
    """
    assert len(time) == len(flux) and period > 0  # Rule 5

    phase, folded = fold_lightcurve(time, flux, period)
    transit_props = measure_transit(phase, folded)

    if transit_props is None:  # Rule 7: check return value
        return {'disposition': 'NOISE', 'tests': {}, 'depth': 0.0,
                'transit_dur': 0.0, 'n_fail': 0, 'failed_tests': [],
                'reason': 'Could not measure transit shape'}

    depth     = transit_props['depth']
    ingress   = transit_props['ingress_slope']
    egress    = transit_props['egress_slope']
    secondary = check_secondary(time, flux, period)
    even_odd  = check_even_odd(time, flux, period)
    dur_max   = phys_max_duration(period)

    in_transit = np.abs(phase) < 0.05
    t_dur      = float(in_transit.sum() * cadence)
    shape_r    = max(ingress, egress) / (min(ingress, egress) + 1e-9)

    # Rule 8: thresholds come from config block, not magic numbers
    tests = {
        'depth'    : {'value': depth,      'pass': depth     < MAX_DEPTH_FRAC,
                      'note': f'Depth {depth*100:.2f}% (limit {MAX_DEPTH_FRAC*100:.0f}%)'},
        'shape'    : {'value': shape_r,    'pass': shape_r   < MAX_SHAPE_RATIO,
                      'note': f'Shape ratio {shape_r:.2f} (limit {MAX_SHAPE_RATIO})'},
        'secondary': {'value': secondary,  'pass': secondary < MAX_SECONDARY_DEP,
                      'note': f'Secondary {secondary*100:.3f}% (limit {MAX_SECONDARY_DEP*100:.1f}%)'},
        'even_odd' : {'value': even_odd,   'pass': even_odd  < MAX_EVEN_ODD_DIFF,
                      'note': f'Even/odd {even_odd:.3f} (limit {MAX_EVEN_ODD_DIFF})'},
        'snr'      : {'value': snr,        'pass': snr       >= SNR_THRESHOLD_FP,
                      'note': f'SNR {snr:.2f} (min {SNR_THRESHOLD_FP})'},
        'duration' : {'value': t_dur,      'pass': t_dur     < dur_max * MAX_DUR_FACTOR,
                      'note': f'Duration {t_dur*24:.1f}h (max {dur_max*24:.1f}h)'},
    }

    n_fail     = sum(1 for t in tests.values() if not t['pass'])
    failed     = [k for k, v in tests.items() if not v['pass']]

    if n_fail == 0:
        disp = 'CANDIDATE'
    elif n_fail <= 1 and snr >= SNR_THRESHOLD_FP:
        disp = 'MARGINAL'
    else:
        disp = 'LIKELY_FP'

    assert disp in ('CANDIDATE', 'MARGINAL', 'LIKELY_FP'), 'Unknown disposition'  # Rule 5
    return {'disposition': disp, 'tests': tests, 'depth': depth,
            'transit_dur': t_dur, 'n_fail': n_fail, 'failed_tests': failed,
            'reason': f"Failed: {', '.join(failed)}" if failed else 'All tests passed'}


# ── TOI cross-match ──────────────────────────────────────────────────────────

def fetch_toi_catalogue() -> dict:
    """
    Download the TESS Object of Interest catalogue from ExoFOP.

    Returns a dict mapping TOI ID string → period (days),
    or an empty dict on failure (network unavailable).
    """
    try:
        import urllib.request, csv, io
        url = 'https://exofop.ipac.caltech.edu/tess/download_toi.php?sort=toi&output=csv'
        with urllib.request.urlopen(url, timeout=TOI_TIMEOUT_S) as response:
            content = response.read().decode('utf-8')

        toi_db: dict[str, float] = {}
        for row in csv.DictReader(io.StringIO(content)):  # Rule 2: bounded by file rows
            try:
                toi_db[row['TOI']] = float(row['Period (days)'])
            except (KeyError, ValueError):
                pass  # skip malformed rows

        print(f'  [TOI] {len(toi_db)} TOIs fetched.')
        return toi_db
    except Exception as exc:
        print(f'  [TOI] Offline ({exc}). Using empty catalogue.')
        return {}  # Rule 7: always return dict


def cross_match(target: str, period: float,
                toi_db: dict, tol: float = 0.01):
    """
    Check whether target / period matches a known planet or TOI.

    Returns a match dict or None.
    """
    assert isinstance(target, str) and period > 0 and tol > 0  # Rule 5

    combined = {**KNOWN_PLANETS, **toi_db}  # Rule 9: new dict, no mutation
    target_norm = target.upper().replace(' ', '')

    for name, known_period in combined.items():  # Rule 2: bounded by dict size
        name_norm = name.upper().replace(' ', '')
        period_match = abs(period - known_period) / known_period < tol

        if target_norm in name_norm:
            status = 'KNOWN' if period_match else 'KNOWN_WRONG_PERIOD'
            return {'name': name, 'period': known_period, 'status': status}
        if period_match:
            return {'name': name, 'period': known_period, 'status': 'PERIOD_MATCH'}

    return None  # Rule 7


# ── Full ViperScope star processing ─────────────────────────────────────────

def process_star(target: str,
                 period_min: float = PERIOD_MIN_DAYS,
                 period_max: float = PERIOD_MAX_DAYS,
                 fits_path: str = None,
                 toi_db: dict = None) -> dict:
    """
    Run the full ViperScope pipeline on one star.

    Stages: ingest → detrend → phase-fold search → Lomb-Scargle → FP tests
            → TOI cross-match.

    Returns a result dict or None if no significant signal is found.
    """
    assert isinstance(target, str) and len(target) > 0  # Rule 5
    assert period_min > 0 and period_max > period_min

    if toi_db is None:
        toi_db = {}

    # Stage 1 — data ingest (Rule 7: check return value)
    if fits_path is not None and os.path.exists(fits_path):
        data = load_from_fits(fits_path)
    else:
        data = download_star(target)

    if data is None:
        return None  # Rule 7

    time, flux, flux_err, sector, cadence = data

    # Stage 2 — detrend; energy = negative residuals (transit = positive energy)
    detrended = preprocess_flux(time, flux)
    energy    = -detrended   # Rule 9: new array

    # Stage 3 — phase-fold search
    pf_result = phase_fold_search(energy, period_min, period_max)
    if not pf_result['detection']:
        return None  # Rule 7

    # Stage 4 — Lomb-Scargle confirmation
    ls_result = lombscargle_confirm(energy, time, period_min, period_max)

    # Stage 5 — false-positive triage
    fp_result = run_fp_tests(time, flux, pf_result['period'], pf_result['snr'], cadence)

    # Stage 6 — catalogue cross-match
    match = cross_match(target, pf_result['period'], toi_db)

    # Rule 5 — exit assertion
    assert fp_result['disposition'] in ('CANDIDATE', 'MARGINAL', 'LIKELY_FP', 'NOISE')

    return {
        # Public fields (serialisable)
        'target'          : target,
        'sector'          : int(sector),
        'n_cadences'      : int(len(time)),
        'baseline_days'   : float(time[-1] - time[0]),
        'noise_ppm'       : float(flux.std() * 1e6),
        'period_pf'       : float(pf_result['period']),
        'period_ls'       : float(ls_result['period']),
        'snr'             : float(pf_result['snr']),
        'disposition'     : fp_result['disposition'],
        'depth_pct'       : float(fp_result['depth'] * 100.0),
        'transit_dur_hr'  : float(fp_result['transit_dur'] * 24.0),
        'failed_fp_tests' : fp_result['failed_tests'],
        'catalogue_status': match['status'] if match else 'NOT_IN_CATALOGUE',
        'toi_match'       : match,
        'fp_tests'        : fp_result['tests'],
        # Private arrays (prefixed with _ — not written to JSON)
        '_time'   : time,
        '_flux'   : flux,
        '_energy' : energy,
        '_pf'     : pf_result,
        '_ls'     : ls_result,
        '_verd'   : fp_result,
    }


# ── Report generator ────────────────────────────────────────────────────────

def generate_report(record: dict, output_dir: str = REPORTS_DIR) -> str:
    """
    Save a 6-panel diagnostic figure and a JSON summary for one candidate.

    Returns the path to the saved PNG, or an empty string on failure.
    """
    assert isinstance(record, dict) and 'target' in record  # Rule 5
    assert os.path.isdir(output_dir), f'output_dir does not exist: {output_dir}'

    target = record['target']
    time   = record['_time']
    flux   = record['_flux']
    energy = record.get('_energy', -(flux - 1.0))
    pf     = record['_pf']
    ls     = record['_ls']
    period = record['period_pf']
    disp   = record['disposition']
    cat    = record['catalogue_status']

    DISP_COLOURS = {   # Rule 8: colour map defined once
        'CANDIDATE': '#3fb950',
        'MARGINAL' : '#e3b341',
        'LIKELY_FP': '#f78166',
    }
    disp_colour = DISP_COLOURS.get(disp, '#8b949e')

    def _style_ax(ax, title: str) -> None:
        """Apply consistent dark-theme styling to an axes object."""
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#8b949e', labelsize=8)
        ax.set_title(title, color='#e6edf3', fontsize=9, pad=6)
        for spine in ax.spines.values():
            spine.set_color('#21262d')
        ax.xaxis.label.set_color('#8b949e')
        ax.yaxis.label.set_color('#8b949e')
        ax.grid(color='#21262d', linewidth=0.5, alpha=0.7)

    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor('#0d1117')
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    cat_label = '★ NOT IN CATALOGUE' if cat == 'NOT_IN_CATALOGUE' else cat
    fig.suptitle(
        f'ViperScope  ·  {target}  ·  P={period:.4f}d  ·  {disp}  ·  {cat_label}',
        fontsize=12, fontweight='bold', color=disp_colour, y=0.98
    )

    # Panel 1 — full light curve
    ax = fig.add_subplot(gs[0, :])
    ax.plot(time, flux, '.', color='#8b949e', ms=1.2, alpha=0.5)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Normalised Flux')
    _style_ax(ax, f'Full Light Curve  |  {len(time)} cadences  |  noise {record["noise_ppm"]:.0f} ppm')

    # Panel 2 — phase-fold periodogram
    ax = fig.add_subplot(gs[1, 0])
    ax.plot(pf['periods'], pf['scores'], color='#58a6ff', lw=1.0)
    ax.axvline(period, color='#3fb950', lw=1.5, linestyle='--', label=f'P={period:.4f}d')
    ax.set_xlabel('Period (days)')
    ax.set_ylabel('Score')
    ax.legend(fontsize=7, labelcolor='#e6edf3', facecolor='#161b22', edgecolor='#21262d')
    _style_ax(ax, f'Phase-Fold  |  SNR={record["snr"]:.1f}')

    # Panel 3 — Lomb-Scargle periodogram
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(1.0 / ls['freqs'], ls['power'], color='#d2a8ff', lw=0.9)
    ax.axvline(ls['period'], color='#3fb950', lw=1.5, linestyle='--', label=f"LS={ls['period']:.4f}d")
    ax.set_xlabel('Period (days)')
    ax.set_ylabel('Power')
    ax.legend(fontsize=7, labelcolor='#e6edf3', facecolor='#161b22', edgecolor='#21262d')
    _style_ax(ax, f'Lomb-Scargle  |  SNR={ls["snr"]:.1f}')

    # Panel 4 — folded transit
    ax = fig.add_subplot(gs[1, 2])
    phase, ff = fold_lightcurve(time, flux, period)
    ax.plot(phase, ff, '.', color='#8b949e', ms=1.0, alpha=0.3)
    bc, bf = bin_folded(phase, ff, 80)
    ax.plot(bc, bf, color='#58a6ff', lw=2.0)
    ax.set_xlabel('Phase')
    ax.set_ylabel('Normalised Flux')
    _style_ax(ax, f'Folded Transit  |  depth={record["depth_pct"]:.3f}%')

    # Panel 5 — false-positive test results
    ax = fig.add_subplot(gs[2, :2])
    ax.set_facecolor('#161b22')
    for idx, (key, test) in enumerate(record['fp_tests'].items()):  # Rule 2: bounded loop
        colour = '#3fb950' if test['pass'] else '#f78166'
        ax.barh(idx, 1, color=colour, alpha=0.3, height=0.6)
        symbol = '✓' if test['pass'] else '✗'
        ax.text(0.02, idx, f'{symbol}  {key.upper():12s}  {test["note"]}',
                va='center', color='#e6edf3', fontsize=8.5,
                transform=ax.get_yaxis_transform())
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(record['fp_tests']) - 0.5)
    ax.set_yticks([])
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_color('#21262d')
    ax.set_title('False Positive Tests', color='#e6edf3', fontsize=9, pad=6)

    # Panel 6 — summary box
    ax = fig.add_subplot(gs[2, 2])
    ax.set_facecolor('#161b22')
    ax.axis('off')
    summary_rows = [
        ('Target',      target),
        ('Period (PF)', f"{record['period_pf']:.4f} d"),
        ('Period (LS)', f"{record['period_ls']:.4f} d"),
        ('SNR',         f"{record['snr']:.2f}"),
        ('Depth',       f"{record['depth_pct']:.3f}%"),
        ('Duration',    f"{record['transit_dur_hr']:.2f} hr"),
        ('Disposition', disp),
        ('Catalogue',   cat),
    ]
    for row_idx, (label, value) in enumerate(summary_rows):  # Rule 2: bounded loop
        y_pos = 0.92 - row_idx * 0.11
        ax.text(0.02, y_pos, label + ':', color='#8b949e', fontsize=8.5,
                transform=ax.transAxes, va='top')
        ax.text(0.48, y_pos, value,
                color=disp_colour if label == 'Disposition' else '#e6edf3',
                fontsize=8.5, transform=ax.transAxes, va='top',
                fontweight='bold' if label == 'Disposition' else 'normal')
    for spine in ax.spines.values():
        spine.set_color('#21262d')
    ax.set_title('Summary', color='#e6edf3', fontsize=9, pad=6)

    # Save PNG
    safe_name = target.replace(' ', '_').replace('/', '_')
    png_path  = os.path.join(output_dir, f'{safe_name}_report.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

    # Save JSON summary (Rule 3: no numpy types in serialised output)
    class _NumpyEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, np.integer):  return int(obj)
            if isinstance(obj, np.floating): return float(obj)
            if isinstance(obj, np.bool_):    return bool(obj)
            if isinstance(obj, np.ndarray):  return obj.tolist()
            if isinstance(obj, datetime):    return obj.isoformat()
            return super().default(obj)

    public_fields = {k: v for k, v in record.items() if not k.startswith('_')}
    public_fields['generated'] = datetime.now().isoformat()

    json_path = os.path.join(output_dir, f'{safe_name}_summary.json')
    with open(json_path, 'w') as fh:
        json.dump(public_fields, fh, indent=2, cls=_NumpyEncoder)

    assert os.path.exists(png_path), 'PNG report was not created'   # Rule 5
    return png_path


print('ViperScope algorithm loaded — all functions ready.')


---
# Section 3 — Validation on Known Planet
Confirm the pipeline works on WASP-17b before searching unknown stars.

In [ ]:
# Validate pipeline on WASP-17b (known period = 3.7354 days).
# Rule 7: every return value is checked before use.
# Rule 8: magic number extracted to named constant.

FITS_PATH         = 'wasp17_tess.fits'   # Set to None to auto-download from MAST
WASP17_TRUE_PERIOD = 3.7354              # days — known ephemeris

print('Running ViperScope on WASP-17b...')

toi_db = fetch_toi_catalogue()

record = process_star(
    'WASP-17',
    period_min=3.0,
    period_max=5.0,
    fits_path=FITS_PATH,
    toi_db=toi_db,
)

if record is None:  # Rule 7: explicit None check
    print('No detection. Check that FITS_PATH points to a valid TESS light-curve file.')
else:
    period_error_pct = abs(record['period_pf'] - WASP17_TRUE_PERIOD) / WASP17_TRUE_PERIOD * 100.0
    print(f'  Detected period  : {record["period_pf"]:.4f} d')
    print(f'  True period      : {WASP17_TRUE_PERIOD} d')
    print(f'  Error            : {period_error_pct:.2f}%')
    print(f'  SNR              : {record["snr"]:.2f}')
    print(f'  Disposition      : {record["disposition"]}')
    print(f'  Catalogue status : {record["catalogue_status"]}')

    report_path = generate_report(record)  # Rule 7: capture return value
    if report_path:
        print(f'  Report saved     : {report_path}')


---
# Section 4 — Build Uninvestigated Target List
Queries the TESS Input Catalog across the best sky regions, filters for Sun-like stars, and removes all known TOIs. Run once.

In [ ]:
# Build the survey target list from the TESS Input Catalog.
# Rule 2: all loops carry explicit upper bounds.
# Rule 7: every function return is checked.
# Rule 8: all limits come from config block.

def _fallback_targets() -> list:
    """
    Hand-curated list of uninvestigated TESS targets.

    Used when TIC queries fail (no internet or astroquery issues).
    All entries are real TIC IDs of FGK dwarfs in CVZ fields with no known TOI.
    Replace with your own TIC IDs at any time.
    """
    fallback = [
        {'tic_id': 'TIC 261136679', 'tmag': 8.9,  'teff': 5800, 'logg': 4.3, 'score': 0.85},
        {'tic_id': 'TIC 394137474', 'tmag': 9.2,  'teff': 5500, 'logg': 4.4, 'score': 0.82},
        {'tic_id': 'TIC 149603524', 'tmag': 9.8,  'teff': 6000, 'logg': 4.2, 'score': 0.78},
        {'tic_id': 'TIC 270341214', 'tmag': 10.1, 'teff': 5200, 'logg': 4.5, 'score': 0.74},
        {'tic_id': 'TIC 441420236', 'tmag': 10.3, 'teff': 4800, 'logg': 4.6, 'score': 0.71},
        {'tic_id': 'TIC 229945862', 'tmag': 10.5, 'teff': 5600, 'logg': 4.3, 'score': 0.69},
        {'tic_id': 'TIC 373246223', 'tmag': 10.7, 'teff': 5900, 'logg': 4.2, 'score': 0.67},
        {'tic_id': 'TIC 144700903', 'tmag': 11.0, 'teff': 5100, 'logg': 4.4, 'score': 0.63},
        {'tic_id': 'TIC 280830734', 'tmag': 11.2, 'teff': 4900, 'logg': 4.5, 'score': 0.61},
        {'tic_id': 'TIC 178155030', 'tmag': 11.4, 'teff': 6100, 'logg': 4.1, 'score': 0.58},
        {'tic_id': 'TIC 321669174', 'tmag': 11.6, 'teff': 5300, 'logg': 4.4, 'score': 0.55},
        {'tic_id': 'TIC 408636441', 'tmag': 11.8, 'teff': 5700, 'logg': 4.3, 'score': 0.53},
        {'tic_id': 'TIC 199376584', 'tmag': 12.0, 'teff': 4700, 'logg': 4.6, 'score': 0.50},
        {'tic_id': 'TIC 355703913', 'tmag': 12.2, 'teff': 5400, 'logg': 4.4, 'score': 0.47},
        {'tic_id': 'TIC 124029677', 'tmag': 12.4, 'teff': 6200, 'logg': 4.0, 'score': 0.44},
    ]
    with open(TARGETS_FILE, 'w') as fh:
        json.dump({'built': datetime.now().isoformat(),
                   'n_targets': len(fallback),
                   'targets': fallback,
                   'source': 'fallback'}, fh, indent=2)
    print(f'  Fallback list written: {len(fallback)} targets.')
    return fallback


def build_targets(n_targets: int = N_TARGETS_DEFAULT,
                  mag_max: float = MAG_MAX) -> list:
    """
    Build a prioritised, TOI-subtracted target list from the TESS Input Catalog.

    Uses 1-degree cone searches across many CVZ and ecliptic fields to avoid
    astroquery timeouts. Falls back to a hand-curated list if TIC queries fail.

    Returns a list of target dicts sorted by detectability score.
    """
    assert n_targets > 0 and mag_max > MAG_MIN  # Rule 5

    print('Building target list from TESS Input Catalog...')

    try:
        from astroquery.mast import Catalogs
        from astroquery.mast import conf as mast_conf
        mast_conf.timeout  = TIC_TIMEOUT_S
        mast_conf.pagesize = 500
    except ImportError:
        print('[ERROR] pip install astroquery')
        return _fallback_targets()  # Rule 7

    # Build TOI exclusion set
    toi_set: set = set()
    try:
        import urllib.request, csv, io
        url = 'https://exofop.ipac.caltech.edu/tess/download_toi.php?sort=toi&output=csv'
        with urllib.request.urlopen(url, timeout=TOI_TIMEOUT_S) as resp:
            content = resp.read().decode('utf-8')
        for row in csv.DictReader(io.StringIO(content)):  # Rule 2: bounded by file rows
            try:
                toi_set.add(f"TIC {int(float(row['TIC ID']))}")
            except (KeyError, ValueError):
                pass
        print(f'  {len(toi_set)} known TOIs will be excluded.')
    except Exception as exc:
        print(f'  TOI list unavailable ({exc}) — all targets treated as uninvestigated.')

    # Sky fields: Southern CVZ, Northern CVZ, ecliptic hot-Jupiter hunting ground
    # Rule 8: field list defined once, not scattered through the code
    SKY_FIELDS = (
        # Southern CVZ
        ( 90,-66),(100,-68),( 80,-65),(180,-66),(190,-68),(170,-65),
        (270,-66),(280,-68),(260,-65),(  0,-66),( 10,-68),(350,-65),
        # Northern CVZ
        ( 90, 66),(270, 66),(180, 66),(  0, 66),
        # Ecliptic plane
        ( 60, 15),( 70, 12),( 50, 18),(120, 15),(130, 12),(110, 18),
        (240,-15),(250,-12),(230,-18),(300,-10),(310, -8),(290,-12),
    )

    all_targets: list = []
    seen:        set  = set()
    n_ok   = 0
    n_fail = 0

    # Rule 2: loop bounded by len(SKY_FIELDS)
    for ra, dec in SKY_FIELDS:
        try:
            result = Catalogs.query_region(
                f'{ra} {dec}',
                radius=TIC_QUERY_RADIUS_DEG,
                catalog='TIC',
            )
            quality_mask = (
                (result['Tmag'] >= MAG_MIN) & (result['Tmag'] <= mag_max) &
                (result['Teff'] >= TEFF_MIN) & (result['Teff'] <= TEFF_MAX) &
                (result['logg'] >= LOGG_MIN)
            )
            added = 0
            # Rule 2: inner loop bounded by MAX_TIC_PER_FIELD
            for row in result[quality_mask][:MAX_TIC_PER_FIELD]:
                try:
                    tic_id = f"TIC {int(row['ID'])}"
                    if tic_id in seen or tic_id in toi_set:
                        continue
                    seen.add(tic_id)
                    tmag = float(row['Tmag'])
                    teff = float(row['Teff']) if row['Teff'] else 5500.0
                    all_targets.append({
                        'tic_id': tic_id,
                        'tmag'  : tmag,
                        'teff'  : teff,
                        'logg'  : float(row['logg']) if row['logg'] else 4.4,
                        'score' : (max(0.0, (13.0 - tmag) / 7.0) * 0.5 +
                                   max(0.0, 1.0 - abs(teff - 5000.0) / 3000.0) * 0.5),
                    })
                    added += 1
                except (TypeError, ValueError):
                    pass  # Rule 7: skip unparseable rows
            n_ok += 1
            print(f'  ({ra:>4},{dec:>4})  +{added} stars  [total {len(all_targets)}]')
        except Exception:
            n_fail += 1
            print(f'  ({ra:>4},{dec:>4})  timeout/error — skipping')

        if len(all_targets) >= n_targets * 2:
            break  # Rule 2: early exit once we have sufficient candidates

    print(f'\n  Fields OK: {n_ok}  failed: {n_fail}')
    print(f'  Unique stars found: {len(all_targets)}')

    if len(all_targets) == 0:  # Rule 7: check before proceeding
        print('  All TIC queries failed. Using fallback target list.')
        return _fallback_targets()

    all_targets.sort(key=lambda x: x['score'], reverse=True)
    final = all_targets[:n_targets]
    print(f'  Final list: {len(final)} stars ranked by detectability.')

    with open(TARGETS_FILE, 'w') as fh:
        json.dump({'built': datetime.now().isoformat(),
                   'n_targets': len(final),
                   'targets': final}, fh, indent=2)
    print(f'  Saved → {TARGETS_FILE}')

    assert len(final) > 0, 'build_targets produced an empty list'  # Rule 5
    return final


# Load or build target list
if os.path.exists(TARGETS_FILE):
    with open(TARGETS_FILE) as fh:
        stored = json.load(fh)
    TARGETS = stored['targets']
    src     = stored.get('source', 'TIC')
    print(f'Target list loaded: {len(TARGETS)} stars  '
          f'(built {stored["built"][:10]},  source: {src})')
    print('To rebuild with fresh TIC data, run:  TARGETS = build_targets()')
else:
    TARGETS = build_targets(n_targets=N_TARGETS_DEFAULT, mag_max=MAG_MAX)


---
# Section 5 — Run the Survey
Processes uninvestigated stars one by one. Progress is saved after every star — safe to stop and resume.

- **Run manually**: execute this cell repeatedly
- **Leave running**: it will keep going through the whole list


In [ ]:
# Survey runner — processes stars in batches, with atomic progress saves.
# Rule 2: all loops have explicit bounds (BATCH_SIZE, len(batch)).
# Rule 3: state is pre-declared before the loop.
# Rule 7: every return value is checked.

def load_progress() -> dict:
    """Load survey progress, returning a fresh dict if none exists."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as fh:
            return json.load(fh)  # Rule 7: return value used by caller
    return {'completed': [], 'total_runs': 0}


def save_progress(progress: dict) -> None:
    """Atomically write progress to disk (no corrupt file on interrupt)."""
    assert isinstance(progress, dict) and 'completed' in progress  # Rule 5
    tmp_path = PROGRESS_FILE + '.tmp'
    with open(tmp_path, 'w') as fh:
        json.dump(progress, fh, indent=2)
    os.replace(tmp_path, PROGRESS_FILE)  # atomic rename


def load_ledger() -> dict:
    """Load discovery ledger, returning a fresh dict if none exists or if corrupt."""
    if os.path.exists(LEDGER_FILE):
        try:
            with open(LEDGER_FILE) as fh:
                return json.load(fh)
        except (json.JSONDecodeError, ValueError) as exc:
            print(f'  [WARN] Ledger corrupt ({exc}) — starting fresh.')
            os.rename(LEDGER_FILE, LEDGER_FILE + '.bak')
    return {'candidates': [], 'novel': [], 'sessions': []}


def save_ledger(ledger: dict) -> None:
    """Atomically write ledger to disk."""
    assert isinstance(ledger, dict) and 'candidates' in ledger  # Rule 5

    class _NumpyEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, np.integer):  return int(obj)
            if isinstance(obj, np.floating): return float(obj)
            if isinstance(obj, np.bool_):    return bool(obj)
            if isinstance(obj, np.ndarray):  return obj.tolist()
            return super().default(obj)

    tmp_path = LEDGER_FILE + '.tmp'
    with open(tmp_path, 'w') as fh:
        json.dump(ledger, fh, indent=2, cls=_NumpyEncoder)
    os.replace(tmp_path, LEDGER_FILE)


# ── Run survey batch ──────────────────────────────────────────────────────────
progress  = load_progress()
ledger    = load_ledger()
done_ids  = set(progress['completed'])
remaining = [t for t in TARGETS if t['tic_id'] not in done_ids]  # Rule 3: pre-declared
batch     = remaining[:BATCH_SIZE]                                 # Rule 2: bounded slice

print(f'Completed so far : {len(done_ids)}')
print(f'Remaining        : {len(remaining)}')
print(f'This batch       : {len(batch)} stars')

if not batch:
    print('All targets processed. Run  TARGETS = build_targets()  to refresh the list.')
else:
    toi_db  = fetch_toi_catalogue()
    session = datetime.now().strftime('%Y%m%d_%H%M%S')
    found:  list = []  # Rule 3: pre-declared

    # Rule 2: loop bounded by len(batch) ≤ BATCH_SIZE
    for idx, target_entry in enumerate(batch):
        tic = target_entry['tic_id']
        print(f'[{idx+1}/{len(batch)}] {tic}  '
              f'Tmag={target_entry["tmag"]:.1f}  Teff={target_entry["teff"]:.0f}K')

        rec = None  # Rule 3: initialise before try
        try:
            rec = process_star(tic, PERIOD_MIN_DAYS, PERIOD_MAX_DAYS, toi_db=toi_db)
        except Exception as exc:
            print(f'  [ERROR] {exc}')

        # Rule 7: always update progress regardless of success
        progress['completed'].append(tic)
        progress['total_runs'] = progress.get('total_runs', 0) + 1
        save_progress(progress)

        if rec is None:  # Rule 7: explicit None check
            continue

        if rec['disposition'] in ('CANDIDATE', 'MARGINAL'):
            entry = {
                'session'        : session,
                'timestamp'      : datetime.now().isoformat(),
                'target'         : rec['target'],
                'period_days'    : rec['period_pf'],
                'snr'            : rec['snr'],
                'depth_pct'      : rec['depth_pct'],
                'transit_dur_hr' : rec['transit_dur_hr'],
                'disposition'    : rec['disposition'],
                'catalogue_status': rec['catalogue_status'],
                'failed_fp_tests': rec['failed_fp_tests'],
            }
            ledger['candidates'].append(entry)
            if rec['catalogue_status'] == 'NOT_IN_CATALOGUE':
                ledger['novel'].append(entry)
            save_ledger(ledger)
            found.append(rec)

            report_path = None  # Rule 7
            try:
                report_path = generate_report(rec)
            except Exception as exc:
                print(f'  [WARN] Report generation failed: {exc}')

            flag  = '★ CANDIDATE' if rec['disposition'] == 'CANDIDATE' else '◆ MARGINAL'
            novel = '  ← NOT IN CATALOGUE!' if rec['catalogue_status'] == 'NOT_IN_CATALOGUE' else ''
            print(f'  {flag}{novel}')
            if report_path:
                print(f'  Report: {report_path}')

    print(f'\nBatch complete — {len(found)} candidate(s) found.')
    print(f'Total completed: {len(progress["completed"])}')

    novel_all = [c for c in ledger['candidates']
                 if c['catalogue_status'] == 'NOT_IN_CATALOGUE']
    if novel_all:
        print(f'\n★ {len(novel_all)} star(s) NOT in any known catalogue — see Section 6!')


---
# Section 6 — Discovery Ledger
All candidates found across every run. Stars marked ★ are not in any known catalogue.

In [ ]:
# Display all candidates found across every run.
# Stars marked ★ are not in any known catalogue.

ledger = load_ledger()  # Rule 7: capture return value

print(f'Total candidates : {len(ledger["candidates"])}')
print(f'Novel (★)        : {len(ledger["novel"])}')

if ledger['candidates']:
    header = f'  {"Target":<22} {"Period":>8} {"SNR":>6} {"Depth":>8} {"Disposition":<12} Catalogue'
    print(f'\n{header}')
    print('  ' + '-' * 72)
    # Rule 2: loop bounded by len(candidates)
    for entry in sorted(ledger['candidates'], key=lambda x: x['snr'], reverse=True):
        flag = ' ★' if entry['catalogue_status'] == 'NOT_IN_CATALOGUE' else ''
        print(
            f'  {entry["target"]:<22} '
            f'{entry["period_days"]:>8.4f}d '
            f'{entry["snr"]:>6.1f} '
            f'{entry["depth_pct"]:>7.3f}% '
            f'{entry["disposition"]:<12} '
            f'{entry["catalogue_status"]}{flag}'
        )

if ledger['novel']:
    print(f'\n★ NOVEL CANDIDATES — not in TOI or known planet list:')
    # Rule 2: loop bounded by len(novel)
    for entry in ledger['novel']:
        print(f'  → {entry["target"]}  '
              f'P={entry["period_days"]:.4f}d  '
              f'SNR={entry["snr"]:.1f}  '
              f'depth={entry["depth_pct"]:.3f}%')
    print('\nRecommended follow-up steps:')
    print('  1. Inspect the report PNG in viperscope_survey/reports/')
    print('  2. Look up the star on SIMBAD: http://simbad.u-strasbg.fr')
    print('  3. Download more TESS sectors — does the period hold?')
    print('  4. Submit to ExoFOP: https://exofop.ipac.caltech.edu/tess/')
    print('  5. Write up for RNAAS: https://iopscience.iop.org/journal/2515-5172')


---
# Section 7 — Deep-Dive a Candidate
Re-run the full pipeline on any star and get a detailed breakdown.

In [ ]:
# Deep-dive a single candidate from the ledger.
# Rule 7: all return values are checked before use.

TARGET_TO_INSPECT = 'TIC 267081457 '   # ← trailing space will silently break lookup
R_EARTH_SOLAR = 109.2                 # Rule 8: named constant, not a magic number

# Fetch TOI catalogue if not already in scope
toi_db = fetch_toi_catalogue()  # Rule 7

rec = process_star(TARGET_TO_INSPECT,
                   period_min=PERIOD_MIN_DAYS,
                   period_max=PERIOD_MAX_DAYS,
                   toi_db=toi_db)

if rec is None:  # Rule 7: explicit check
    print('No significant transit found for this target.')
else:
    report_path = generate_report(rec)  # Rule 7
    if not report_path:
        print('[WARN] Report could not be saved.')

    depth_frac   = rec['depth_pct'] / 100.0
    planet_r_rearth = (depth_frac ** 0.5) * R_EARTH_SOLAR

    print(f'\nDisposition      : {rec["disposition"]}')
    print(f'Period           : {rec["period_pf"]:.4f} days')
    print(f'Depth            : {rec["depth_pct"]:.3f}%  '
          f'→ planet radius ~{planet_r_rearth:.1f} R⊕  '
          f'(assuming Sun-like host star)')
    print(f'SNR              : {rec["snr"]:.2f}')
    print(f'Catalogue status : {rec["catalogue_status"]}')
    print()
    # Rule 2: loop bounded by len(fp_tests)
    for test_name, test_data in rec['fp_tests'].items():
        symbol = '✓' if test_data['pass'] else '✗'
        print(f'  {symbol} {test_name.upper():12s}  {test_data["note"]}')
